# VLM + SAM3 商品示例分割诊断实验

这个 notebook 用于验证“多品类 reference exemplar -> production image 目标实例分割”的诊断链路。核心思路是：先用 VLM 根据参考图自动生成类别和文本 prompts，再用 SAM3 text prompt 召回候选 mask，最后用可替换的视觉 embedding 模型对每个候选和对应 target 的 reference 做相似度排序。

推荐数据结构：

```text
data/references/<target_id>/*.png|jpg
data/production/*.png|jpg
```

`data/references/` 根目录下直接放图片也可以运行，notebook 会临时把它们视为一个单独目标 `references_root`。但多品类实验建议使用每个 target 一个目录，否则鞋、帽子、包、衣服会被混成同一个 reference 集合。


In [ ]:
from pathlib import Path
import base64
import csv
import io
import json
import os
import re
import sys
import traceback
from contextlib import nullcontext

import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

try:
    import torch
except Exception as e:
    raise RuntimeError('PyTorch 导入失败。请确认当前 kernel 使用的是 SAM3 对应的 notebook 环境。') from e

# ===== 实验参数 =====
DEMO_REPO = Path('/home/sunpcm/workspace/sam3_demo').resolve()
SAM3_CODE_REPO = Path('/home/sunpcm/workspace/sam3').resolve()
OUTPUT_DIR = DEMO_REPO / 'outputs' / 'vlm_sam3_diagnostic'

REFERENCE_DIR = DEMO_REPO / 'data' / 'references'
PRODUCTION_DIR = DEMO_REPO / 'data' / 'production'
BPE_PATH = SAM3_CODE_REPO / 'sam3' / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'

VLM_BASE_URL = os.getenv('VLM_BASE_URL')
VLM_API_KEY = os.getenv('VLM_API_KEY')
VLM_MODEL = os.getenv('VLM_MODEL')
VLM_FORCE_REFRESH = os.getenv('VLM_FORCE_REFRESH', '0') == '1'

TEXT_PROMPTS_PER_TARGET_LIMIT = int(os.getenv('TEXT_PROMPTS_PER_TARGET_LIMIT', '6'))
EMBEDDING_MODEL_NAME = os.getenv('EMBEDDING_MODEL_NAME', 'facebook/dinov2-base')
DINO_MODEL_NAME = EMBEDDING_MODEL_NAME  # backward-compatible alias
TOP_K_DISPLAY = int(os.getenv('TOP_K_DISPLAY', '12'))
MAX_REFERENCE_IMAGES_FOR_VLM = int(os.getenv('MAX_REFERENCE_IMAGES_FOR_VLM', '4'))
MAX_SAM3_PROMPTS_PER_TARGET = int(os.getenv('MAX_SAM3_PROMPTS_PER_TARGET', str(TEXT_PROMPTS_PER_TARGET_LIMIT)))

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAM3_AMP_DTYPE = torch.bfloat16

os.chdir(DEMO_REPO)
for repo in [DEMO_REPO, SAM3_CODE_REPO]:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

for subdir in ['reference_clean', 'vlm_prompts', 'candidates', 'rankings']:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print('python:', sys.executable)
print('torch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('DEMO_REPO:', DEMO_REPO)
print('SAM3_CODE_REPO:', SAM3_CODE_REPO, '存在=', SAM3_CODE_REPO.exists())
print('BPE_PATH:', BPE_PATH, '存在=', BPE_PATH.exists())
print('REFERENCE_DIR:', REFERENCE_DIR, '存在=', REFERENCE_DIR.exists())
print('PRODUCTION_DIR:', PRODUCTION_DIR, '存在=', PRODUCTION_DIR.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)
print('VLM_BASE_URL 已设置:', bool(VLM_BASE_URL))
print('VLM_API_KEY 已设置:', bool(VLM_API_KEY))
print('VLM_MODEL:', VLM_MODEL)
print('EMBEDDING_MODEL_NAME:', EMBEDDING_MODEL_NAME)

if not REFERENCE_DIR.exists():
    raise FileNotFoundError(f'找不到 reference 目录: {REFERENCE_DIR}')
if not PRODUCTION_DIR.exists():
    raise FileNotFoundError(f'找不到 production 目录: {PRODUCTION_DIR}')
if not SAM3_CODE_REPO.exists():
    raise FileNotFoundError(f'找不到 SAM3 代码仓库: {SAM3_CODE_REPO}')
if not BPE_PATH.exists():
    raise FileNotFoundError(f'找不到 SAM3 BPE 文件: {BPE_PATH}')


## 数据扫描与 Reference 清洗

这一节会忽略隐藏文件和 `.ipynb_checkpoints`。reference 图会先清洗成白底 crop，再用于 VLM prompt 生成和 embedding 提取。


In [ ]:
def is_hidden_path(path):
    return any(part.startswith('.') or part == '__MACOSX' for part in Path(path).parts)

def is_image_path(path):
    path = Path(path)
    return path.suffix.lower() in IMAGE_EXTS and not is_hidden_path(path)

def list_images(directory):
    directory = Path(directory)
    if not directory.exists():
        return []
    return sorted([p for p in directory.rglob('*') if p.is_file() and is_image_path(p)])

def discover_targets(reference_dir):
    reference_dir = Path(reference_dir)
    targets = {}
    direct_images = sorted([p for p in reference_dir.iterdir() if p.is_file() and is_image_path(p)])
    for child in sorted(reference_dir.iterdir()):
        if child.is_dir() and not is_hidden_path(child):
            images = list_images(child)
            if images:
                targets[child.name] = images
    if direct_images:
        targets['references_root'] = direct_images
        print('警告：发现 data/references 根目录下直接放置的 reference 图片，将临时使用 target_id=references_root。')
    return targets

def clean_reference_image(src_path, dst_path, alpha_threshold=8, padding=8):
    img = Image.open(src_path)
    if img.mode in ('RGBA', 'LA') or ('transparency' in img.info):
        rgba = img.convert('RGBA')
        arr = np.array(rgba)
        alpha = arr[..., 3]
        ys, xs = np.where(alpha > alpha_threshold)
        if len(xs) > 0 and len(ys) > 0:
            x1 = max(0, int(xs.min()) - padding)
            y1 = max(0, int(ys.min()) - padding)
            x2 = min(rgba.width, int(xs.max()) + 1 + padding)
            y2 = min(rgba.height, int(ys.max()) + 1 + padding)
            rgba = rgba.crop((x1, y1, x2, y2))
        bg = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
        bg.alpha_composite(rgba)
        clean = bg.convert('RGB')
    else:
        clean = img.convert('RGB')
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    clean.save(dst_path, quality=95)
    return dst_path

targets = discover_targets(REFERENCE_DIR)
production_paths = list_images(PRODUCTION_DIR)

if not targets:
    raise RuntimeError(f'没有在 reference 目录中发现目标图片: {REFERENCE_DIR}')
if not production_paths:
    raise RuntimeError(f'没有在 production 目录中发现生产图: {PRODUCTION_DIR}')

clean_reference_paths = {}
for target_id, paths in targets.items():
    clean_reference_paths[target_id] = []
    for idx, path in enumerate(paths):
        out_path = OUTPUT_DIR / 'reference_clean' / target_id / f'{idx:03d}_{path.stem}.jpg'
        clean_reference_paths[target_id].append(clean_reference_image(path, out_path))

print('识别到的 targets:')
for target_id, paths in targets.items():
    print(f'  {target_id}: raw={len(paths)} clean={len(clean_reference_paths[target_id])}')
print('生产图数量:', len(production_paths))
for p in production_paths:
    print(' ', p.name)

first_target = next(iter(clean_reference_paths))
display(Markdown(f'### 清洗后的 reference 预览：`{first_target}`'))
for p in clean_reference_paths[first_target][:6]:
    img = Image.open(p).convert('RGB')
    thumb = img.copy()
    thumb.thumbnail((180, 180))
    display(thumb)


## VLM Prompt 生成

这一节使用 OpenAI-compatible SDK，通过 `VLM_BASE_URL`、`VLM_API_KEY` 和 `VLM_MODEL` 连接你配置的 VLM。若缓存 JSON 已存在，默认直接复用；设置 `VLM_FORCE_REFRESH=1` 才会重新请求 VLM。

手工 mock cache 路径示例：`outputs/vlm_sam3_diagnostic/vlm_prompts/<target_id>.json`。


In [ ]:
def image_to_data_url(path, max_side=768):
    img = Image.open(path).convert('RGB')
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    encoded = base64.b64encode(buf.getvalue()).decode('ascii')
    return 'data:image/jpeg;base64,' + encoded

def extract_json_object(text):
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', text, flags=re.S)
    if not match:
        raise ValueError('VLM 返回内容里没有可解析的 JSON 对象')
    return json.loads(match.group(0))

def normalize_prompt_payload(payload, target_id):
    def listify(value):
        if value is None:
            return []
        if isinstance(value, str):
            return [value]
        if isinstance(value, list):
            return [str(x).strip() for x in value if str(x).strip()]
        return [str(value).strip()]

    category = str(payload.get('category') or 'product').strip()
    prompts = listify(payload.get('prompts'))
    negative_prompts = listify(payload.get('negative_prompts'))
    attributes = listify(payload.get('attributes'))
    if category and category.lower() not in {p.lower() for p in prompts}:
        prompts = [category] + prompts
    deduped = []
    seen = set()
    for prompt in prompts:
        key = prompt.lower()
        if prompt and key not in seen:
            deduped.append(prompt)
            seen.add(key)
    return {
        'target_id': target_id,
        'category': category,
        'prompts': deduped[:TEXT_PROMPTS_PER_TARGET_LIMIT],
        'negative_prompts': negative_prompts,
        'attributes': attributes,
    }

def call_vlm_for_prompts(target_id, reference_paths):
    if not (VLM_BASE_URL and VLM_API_KEY and VLM_MODEL):
        raise RuntimeError(
            '缺少 VLM 环境变量，且没有可复用的 cache。请设置 VLM_BASE_URL、VLM_API_KEY、VLM_MODEL，'
            '或者手工创建 outputs/vlm_sam3_diagnostic/vlm_prompts/<target_id>.json。'
        )
    from openai import OpenAI
    client = OpenAI(api_key=VLM_API_KEY, base_url=VLM_BASE_URL)
    content = [{
        'type': 'text',
        'text': (
            'You are helping an apparel/product vision diagnostic pipeline. '
            'Given several clean reference images of the same target product/SKU, identify the product category and produce short SAM-style text prompts. '
            'Return JSON only with keys: target_id, category, prompts, negative_prompts, attributes. '
            'Use English prompts. Prompts should be category-level phrases such as sneaker, running shoe, baseball cap, backpack, jacket. '
            'Do not include explanations.'
        )
    }]
    for path in reference_paths[:MAX_REFERENCE_IMAGES_FOR_VLM]:
        content.append({'type': 'image_url', 'image_url': {'url': image_to_data_url(path)}})
    content.append({'type': 'text', 'text': f'target_id: {target_id}'})
    response = client.chat.completions.create(
        model=VLM_MODEL,
        messages=[{'role': 'user', 'content': content}],
        temperature=0,
    )
    raw = response.choices[0].message.content
    return normalize_prompt_payload(extract_json_object(raw), target_id)

def load_or_create_prompt_payload(target_id, reference_paths):
    cache_path = OUTPUT_DIR / 'vlm_prompts' / f'{target_id}.json'
    if cache_path.exists() and not VLM_FORCE_REFRESH:
        payload = json.loads(cache_path.read_text())
        payload = normalize_prompt_payload(payload, target_id)
        print(f'已加载 VLM prompt cache: {cache_path}')
    else:
        payload = call_vlm_for_prompts(target_id, reference_paths)
        cache_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
        print(f'已写入 VLM prompt cache: {cache_path}')
    return payload

vlm_prompts = {}
for target_id, paths in clean_reference_paths.items():
    vlm_prompts[target_id] = load_or_create_prompt_payload(target_id, paths)

display(Markdown('### VLM prompt 结果'))
for target_id, payload in vlm_prompts.items():
    print(json.dumps(payload, indent=2, ensure_ascii=False))


## SAM3 候选生成

这一节使用 SAM3 作为第一版候选召回器。所有候选都会被保存，哪怕后续 embedding 相似度很低，方便判断问题到底出在 prompt、SAM3 召回，还是排序阶段。后续可以在 SAM 前接入 LocateAnything，形成 `text grounding -> bbox/point -> SAM mask` 分支。


In [ ]:
try:
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
except Exception as e:
    raise RuntimeError('无法导入 SAM3。请确认已安装本地 SAM3 repo，并使用对应 notebook kernel。') from e

def sam3_amp_context():
    if DEVICE == 'cuda':
        return torch.autocast(device_type='cuda', dtype=SAM3_AMP_DTYPE)
    return nullcontext()

def tensor_to_numpy(x):
    if isinstance(x, torch.Tensor):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    return np.asarray(x)

def normalize_masks(masks, image_size):
    arr = tensor_to_numpy(masks)
    if arr.ndim == 4:
        if arr.shape[1] == 1:
            arr = arr[:, 0]
        elif arr.shape[-1] == 1:
            arr = arr[..., 0]
    elif arr.ndim == 2:
        arr = arr[None, ...]
    target_w, target_h = image_size
    out = []
    for m in arr:
        m = m > 0.5
        if m.shape != (target_h, target_w):
            m_img = Image.fromarray(m.astype(np.uint8) * 255)
            m_img = m_img.resize((target_w, target_h), resample=Image.NEAREST)
            m = np.array(m_img) > 127
        out.append(m)
    return out

def normalize_boxes(boxes, image_size):
    arr = tensor_to_numpy(boxes).astype(float)
    if arr.size == 0:
        return []
    if arr.ndim == 1:
        arr = arr[None, :]
    w, h = image_size
    out = []
    for b in arr:
        b = b[:4].tolist()
        if max(b) <= 1.5:
            x1, y1, x2, y2 = b[0] * w, b[1] * h, b[2] * w, b[3] * h
        else:
            x1, y1, x2, y2 = b
        x1, y1, x2, y2 = int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        out.append((x1, y1, x2, y2))
    return out

def mask_to_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return (int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1))

def overlay_mask(image, mask, bbox=None, color=(255, 0, 0), alpha=0.45):
    arr = np.array(image.convert('RGB')).astype(np.float32)
    arr[mask] = arr[mask] * (1 - alpha) + np.array(color, dtype=np.float32) * alpha
    out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if bbox is not None:
        draw = ImageDraw.Draw(out)
        draw.rectangle(bbox, outline=color, width=3)
    return out

def crop_bbox(image, bbox):
    return image.convert('RGB').crop(bbox)

def crop_with_mask_white_bg(image, mask, bbox):
    arr = np.array(image.convert('RGB'))
    x1, y1, x2, y2 = bbox
    crop = arr[y1:y2, x1:x2]
    m = mask[y1:y2, x1:x2]
    bg = np.full_like(crop, 255, dtype=np.uint8)
    bg[m] = crop[m]
    return Image.fromarray(bg)

print('正在加载 SAM3，设备:', DEVICE)
with sam3_amp_context():
    sam3_model = build_sam3_image_model(
        bpe_path=str(BPE_PATH),
        device=DEVICE,
        eval_mode=True,
        load_from_HF=True,
    )
    processor = Sam3Processor(sam3_model)
print('SAM3 已加载:', type(sam3_model))


In [ ]:
def run_sam3_text_prompts(image, prompts, target_id, image_path):
    all_candidates = []
    for prompt in prompts[:MAX_SAM3_PROMPTS_PER_TARGET]:
        try:
            with sam3_amp_context():
                state = processor.set_image(image)
                output = processor.set_text_prompt(state=state, prompt=prompt)
        except Exception:
            print(f'SAM3 text prompt 调用失败 target={target_id!r} prompt={prompt!r}')
            traceback.print_exc()
            raise

        masks = normalize_masks(output['masks'], image.size)
        boxes = normalize_boxes(output.get('boxes', []), image.size) if 'boxes' in output else []
        scores = tensor_to_numpy(output.get('scores', np.ones(len(masks), dtype=np.float32))).reshape(-1).tolist()
        if not boxes or len(boxes) != len(masks):
            boxes = [mask_to_bbox(m) for m in masks]

        print(f'图片={Path(image_path).name} target={target_id} prompt={prompt!r} mask数量={len(masks)}')
        for i, mask in enumerate(masks):
            bbox = boxes[i]
            if bbox is None:
                continue
            x1, y1, x2, y2 = bbox
            if x2 <= x1 or y2 <= y1:
                continue
            all_candidates.append({
                'image_path': str(image_path),
                'image_stem': Path(image_path).stem,
                'target_id': target_id,
                'category': vlm_prompts[target_id].get('category'),
                'prompt': prompt,
                'local_index': i,
                'mask': mask,
                'bbox': bbox,
                'sam3_score': float(scores[i]) if i < len(scores) else None,
                'source': 'sam3_text',
            })
    return all_candidates

def save_candidate_debug(image, candidate, global_index):
    image_stem = candidate['image_stem']
    target_id = candidate['target_id']
    bbox = candidate['bbox']
    mask = candidate['mask']
    base_dir = OUTPUT_DIR / 'candidates' / image_stem / target_id
    base_dir.mkdir(parents=True, exist_ok=True)
    prefix = f'{global_index:05d}_{candidate["source"]}_{candidate["local_index"]:03d}'
    bbox_path = base_dir / f'{prefix}_bbox.jpg'
    masked_path = base_dir / f'{prefix}_masked_white.jpg'
    overlay_path = base_dir / f'{prefix}_overlay.jpg'
    crop_bbox(image, bbox).save(bbox_path, quality=95)
    crop_with_mask_white_bg(image, mask, bbox).save(masked_path, quality=95)
    overlay_mask(image, mask, bbox=bbox).save(overlay_path, quality=95)
    x1, y1, x2, y2 = bbox
    area = int(mask.sum())
    aspect_ratio = float((x2 - x1) / max(1, y2 - y1))
    return {
        'candidate_id': prefix,
        'image_path': candidate['image_path'],
        'image_stem': image_stem,
        'target_id': target_id,
        'category': candidate['category'],
        'prompt': candidate['prompt'],
        'source': candidate['source'],
        'local_index': candidate['local_index'],
        'sam3_score': candidate['sam3_score'],
        'bbox': list(map(int, bbox)),
        'area': area,
        'aspect_ratio': aspect_ratio,
        'bbox_crop_path': str(bbox_path),
        'masked_crop_path': str(masked_path),
        'overlay_path': str(overlay_path),
    }

all_candidates = []
candidate_rows = []
global_index = 0

for image_path in production_paths:
    image = Image.open(image_path).convert('RGB')
    for target_id, payload in vlm_prompts.items():
        prompts = payload.get('prompts', [])
        candidates = run_sam3_text_prompts(image, prompts, target_id, image_path)
        for candidate in candidates:
            row = save_candidate_debug(image, candidate, global_index)
            candidate['candidate_id'] = row['candidate_id']
            candidate['masked_crop_path'] = row['masked_crop_path']
            candidate['overlay_path'] = row['overlay_path']
            candidate_rows.append(row)
            all_candidates.append(candidate)
            global_index += 1

candidate_manifest_path = OUTPUT_DIR / 'candidates' / 'candidate_manifest.json'
candidate_manifest_path.write_text(json.dumps(candidate_rows, indent=2, ensure_ascii=False))
print('候选总数:', len(all_candidates))
print('候选 manifest:', candidate_manifest_path)

counts = {}
for row in candidate_rows:
    counts[(row['image_stem'], row['target_id'])] = counts.get((row['image_stem'], row['target_id']), 0) + 1
for image_path in production_paths:
    for target_id in vlm_prompts:
        print(f'{image_path.stem} / {target_id}: {counts.get((image_path.stem, target_id), 0)} 个候选')


## Embedding 排序

默认使用 `facebook/dinov2-base`。如果要实验 DINOv3 或 SigLIP 风格模型，可以在启动 Jupyter 前设置 `EMBEDDING_MODEL_NAME`。只要该模型能通过 `transformers.AutoModel` 产出 image embedding，就可以复用这套 ranking 逻辑。


In [ ]:
from transformers import AutoImageProcessor, AutoModel

def extract_image_embedding(model, image_processor, image):
    inputs = image_processor(images=image.convert('RGB'), return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
        feat = outputs.pooler_output
    elif hasattr(outputs, 'last_hidden_state'):
        feat = outputs.last_hidden_state[:, 0]
    elif isinstance(outputs, dict) and 'image_embeds' in outputs:
        feat = outputs['image_embeds']
    else:
        raise RuntimeError(f'暂不支持该 embedding 模型输出类型: {type(outputs)}')
    feat = torch.nn.functional.normalize(feat.float(), dim=-1)
    return feat[0].detach().cpu().numpy()

def mean_normalized(vectors):
    arr = np.stack(vectors).astype(np.float32)
    mean = arr.mean(axis=0)
    norm = np.linalg.norm(mean) + 1e-12
    return mean / norm

def cosine(a, b):
    return float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-12) * (np.linalg.norm(b) + 1e-12)))

print('正在加载 embedding 模型:', EMBEDDING_MODEL_NAME)
image_processor = AutoImageProcessor.from_pretrained(EMBEDDING_MODEL_NAME)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME).to(DEVICE).eval()

reference_embeddings = {}
for target_id, paths in clean_reference_paths.items():
    feats = []
    for path in paths:
        feats.append(extract_image_embedding(embedding_model, image_processor, Image.open(path).convert('RGB')))
    reference_embeddings[target_id] = mean_normalized(feats)
    print(f'reference embedding 已构建: {target_id} n={len(feats)} dim={reference_embeddings[target_id].shape[0]}')

ranking_rows = []
candidate_by_id = {c['candidate_id']: c for c in all_candidates}
for row in candidate_rows:
    target_id = row['target_id']
    crop = Image.open(row['masked_crop_path']).convert('RGB')
    emb = extract_image_embedding(embedding_model, image_processor, crop)
    sim = cosine(emb, reference_embeddings[target_id])
    ranked = dict(row)
    ranked['embedding_model'] = EMBEDDING_MODEL_NAME
    ranked['similarity'] = sim
    ranking_rows.append(ranked)

ranking_rows.sort(key=lambda r: (r['target_id'], r['image_stem'], -r['similarity']))
rank_json_path = OUTPUT_DIR / 'rankings' / 'candidate_ranking.json'
rank_csv_path = OUTPUT_DIR / 'rankings' / 'candidate_ranking.csv'
rank_json_path.write_text(json.dumps(ranking_rows, indent=2, ensure_ascii=False))
if ranking_rows:
    with rank_csv_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(ranking_rows[0].keys()))
        writer.writeheader()
        writer.writerows(ranking_rows)
else:
    rank_csv_path.write_text('')
print('ranking 行数:', len(ranking_rows))
print('ranking json:', rank_json_path)
print('ranking csv:', rank_csv_path)


## Top 候选展示

这一节用来快速判断失败原因：是 VLM prompts 不合适、SAM3 没有召回目标，还是 embedding 排序把目标排低了。即使某个 target/image 没有候选，也可以在上一节的候选数量统计里看到。


In [ ]:
if not ranking_rows:
    display(Markdown('没有生成任何候选。请优先检查 VLM prompts 和 SAM3 text-prompt 召回。'))
else:
    for target_id in vlm_prompts:
        display(Markdown(f'### 目标 `{target_id}`'))
        subset = [r for r in ranking_rows if r['target_id'] == target_id]
        subset.sort(key=lambda r: -r['similarity'])
        if not subset:
            display(Markdown('这个 target 没有候选。'))
            continue
        for rank, row in enumerate(subset[:TOP_K_DISPLAY], start=1):
            display(Markdown(
                f'**#{rank}** 图片=`{row["image_stem"]}` prompt=`{row["prompt"]}` '
                f'相似度=`{row["similarity"]:.4f}` SAM3分数=`{row["sam3_score"]}` bbox=`{row["bbox"]}`'
            ))
            overlay = Image.open(row['overlay_path']).convert('RGB')
            overlay.thumbnail((520, 520))
            display(overlay)

display(Markdown(f'调试产物已保存到 `{OUTPUT_DIR}`'))
